In [1]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import col

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MMDS") \
    .master("local[*]") \
    .config("spark.local.dir", "/Users/oleksandr/spark-tmp") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/28 14:57:10 WARN Utils: Your hostname, MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.234 instead (on interface en0)
25/12/28 14:57:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/28 14:57:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/28 14:57:10 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


## Define schema and dataframes

In [3]:
schema_ratings = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("item_id", IntegerType(), False),
    StructField("rating", IntegerType(), False),
    StructField("timestamp", IntegerType(), False)
])

schema_movies = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("title", StringType(), False),
    StructField('genres', StringType(), False)
])

schema_users = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("gender", StringType(), False),
    StructField('age', StringType(), False),
    StructField('occupation', IntegerType(), False),
    StructField('zip_code', StringType(), False)
])

In [ ]:
train_ratings = spark.read.option("delimiter", "::").csv("../data/ratings_train.dat", schema=schema_ratings)
test_ratings = spark.read.option("delimiter", "::").csv("../data/ratings_test.dat", schema=schema_ratings)

In [ ]:
movies = spark.read.option("delimiter", "::").csv("../data/movies.dat", schema=schema_movies)
movies = (
    movies
    .withColumn("year", substring("title", -5, 4).cast("int"))
    .withColumn("title", substring("title", 0, length("title") - 6))
    .withColumn("genres", split("genres", r"\|"))
)
movies.printSchema()

root
 |-- item_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- year: integer (nullable = true)



In [ ]:
users = spark.read.option("delimiter", "::").csv("../data/users.dat", schema=schema_users)

users = (
    users
    .withColumn("gender", when(col("gender") == "F", 0).otherwise(1))
)

## Movies profile

In [7]:
from pyspark.sql.functions import min, max

year_stats = movies.agg(
    min("year").alias("min_year"),
    max("year").alias("max_year")
).collect()[0]

min_year, max_year = year_stats["min_year"], year_stats["max_year"]

movies = movies.withColumn(
    "year_norm",
    (col("year") - min_year) / (max_year - min_year)
)

In [ ]:
from pyspark.ml.functions import array_to_vector

embeddings_df = (
    spark.read
    .option("header", True)
    .csv("../data/movies_overview_embeddings.csv")
)

emb_cols = [f"emb_{i}" for i in range(64)]

embeddings_df = (
    embeddings_df
    .select(
        col("movieId").cast("int").alias("item_id"),
        *[col(c).cast("float") for c in emb_cols]
    )
    .withColumn(
        "overview_emb_array",
        array(*emb_cols)
    )
)

embeddings_df = embeddings_df.withColumn(
    "overview_emb",
    array_to_vector(col("overview_emb_array"))
).select("item_id", "overview_emb")

movies = movies.join(
    embeddings_df,
    on="item_id",
    how="left"
)

In [ ]:
import re
from pyspark.ml.feature import (
    CountVectorizer,
    VectorAssembler,
    Normalizer
)

def safe_col(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]", "_", name)

movies_enriched = (
    spark.read
    .option("header", True)
    .option("escape", '"')
    .csv("../data/movies_enriched.csv")
    .select(
        col("movieId").alias("item_id"),
        split(col("actors"), r"\|").alias("actors")
    )
)

top_actors = (
    movies_enriched
    .select(explode("actors").alias("actor"))
    .groupBy("actor")
    .count()
    .orderBy(desc("count"))
    .limit(30)
)

top_actors_list = [r["actor"] for r in top_actors.collect()]

actor_cols = []

for actor in top_actors_list:
    col_name = f"actor_{safe_col(actor)}"

    movies_enriched = movies_enriched.withColumn(
        col_name,
        when(array_contains(col("actors"), actor), 1.0).otherwise(0.0)
    )

    actor_cols.append(col_name)

actors_features = movies_enriched.select(
    "item_id",
    *actor_cols
)

movies = (
    movies
    .join(actors_features, on="item_id", how="left")
)

movies = movies.fillna(0.0, subset=actor_cols)

In [10]:
cv = CountVectorizer(
    inputCol="genres",
    outputCol="tf",
    vocabSize=18,
    minDF=1,
    binary=True
)

cv_model = cv.fit(movies)
tf_df = cv_model.transform(movies)

assembler = VectorAssembler(
    inputCols=["tf", "year_norm", "overview_emb"] + actor_cols,
    outputCol="features_raw",
    handleInvalid='skip'
)

movie_features = assembler.transform(tf_df)

normalizer = Normalizer(
    inputCol="features_raw",
    outputCol="features_norm",
    p=2
)

movies_profiles = normalizer.transform(movie_features)

movies_profiles.select(
    "item_id",
    "features_norm"
)

25/12/28 14:58:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[item_id: int, features_norm: vector]

## Users profile

In [11]:
from pyspark.ml.stat import Summarizer
from pyspark.ml.feature import Normalizer

movie_vecs = movies_profiles.select(
    col("item_id"),
    col("features_norm")
)

user_movie_vectors = (
    train_ratings
    .join(broadcast(movie_vecs), on='item_id') ## broadcase here so that movie_vecs is moved to every executor, instead of shuffling ratings
    .select("user_id", "rating", "features_norm")
)

user_profiles = (
    user_movie_vectors
    .groupBy("user_id")
    .agg(
        Summarizer.mean(
            col("features_norm"),
            weightCol=col("rating")
        ).alias("user_features")
    )
)

normalizer = Normalizer(
    inputCol="user_features",
    outputCol="user_features_norm",
    p=2
)

user_profiles = normalizer.transform(user_profiles)

## LSH

In [12]:
from pyspark.ml.feature import BucketedRandomProjectionLSH

lsh = BucketedRandomProjectionLSH(
    inputCol="features_norm",
    outputCol="hashes",
    bucketLength=2.0,
    numHashTables=5
)

lsh_model = lsh.fit(movies_profiles)

movies_lsh = movies_profiles.select(
    col("item_id"),
    col("features_norm")
).cache()

users_lsh = user_profiles.select(
    col("user_id"),
    col("user_features_norm").alias("features_norm")
).cache()


25/12/28 14:58:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [13]:
from pyspark import StorageLevel

recommendations = lsh_model.approxSimilarityJoin(
    users_lsh,
    movies_lsh, 
    threshold=1.5,
    distCol="distance"
).select(
    col("datasetA.user_id").alias("user_id"),
    col("datasetB.item_id").alias("item_id"),
    col("distance")
).persist(StorageLevel.MEMORY_AND_DISK)

recommendations.count()

22923804

In [14]:
from pyspark.sql.window import Window

already_rated = train_ratings.select("user_id", "item_id")

recommendations = recommendations.join(
    already_rated,
    on=["user_id", "item_id"],
    how="left_anti"
)

window = Window.partitionBy("user_id").orderBy(col("distance").asc())
top_k = 1000

ranked_recs = (
    recommendations
    .withColumn("rank", row_number().over(window))
    .filter(col("rank") <= top_k)
)

In [15]:
rated_in_test = (
    test_ratings
    .select("user_id", "item_id", "rating")
)

recommendations_on_rated_in_test = (
    ranked_recs
    .join(rated_in_test, on=["user_id", "item_id"], how="inner")
)

eval_df = (
    recommendations_on_rated_in_test
    .withColumn("relevant", (col("rating") >= 4).cast("int"))
)

user_metrics = (
    eval_df
    .groupBy("user_id")
    .agg(
        (sum("relevant") / lit(top_k)).alias("precision"),
        sum("relevant").alias("hits"),
    )
    .join(
        rated_in_test
        .filter(col("rating") >= 4)
        .groupBy("user_id")
        .count()
        .withColumnRenamed("count", "total_relevant"),
        on="user_id",
        how="left"
    )
    .fillna(0)
    .withColumn(
        "recall",
        when(col("total_relevant") > 0,
             col("hits") / col("total_relevant"))
        .otherwise(lit(0))
    )
)

avg_metrics = user_metrics.agg(
    avg("precision").alias(f"avg_precision@{top_k}"),
    avg("recall").alias(f"avg_recall@{top_k}")
)

avg_metrics.show()

+--------------------+-------------------+
|  avg_precision@1000|    avg_recall@1000|
+--------------------+-------------------+
|0.006249087432643806|0.44391497929701174|
+--------------------+-------------------+

